# tamserve: name what to track

Finds everything you name on two videos, played as cameras, and follows it. No boxes and no clicks: a `Tracker` takes words.

Before you run it:

1. Runtime → Change runtime type → a GPU (the free T4 works).
2. The model that finds things, Meta's SAM 3, is gated. Accept its licence on the [facebook/sam3](https://huggingface.co/facebook/sam3) page, and make a read token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Give the notebook the token: the key icon on the left → add a secret named `HF_TOKEN`. Or run the first cell and paste the token when it asks.

Then Runtime → Run all. It downloads SAM 3, which is 3.5 GB, and takes about five minutes on the free T4.

In [ ]:
!pip install -q "tamserve[sam3]" opencv-python-headless
import huggingface_hub
import torch

if not huggingface_hub.get_token():  # no HF_TOKEN secret: paste a token here
    huggingface_hub.login()
print(torch.cuda.get_device_name())

Two 10-second clips from [Pexels](https://www.pexels.com), cut to 1280 x 720 and attached to the tamserve release; free to use under the [Pexels licence](https://www.pexels.com/license/). Originals: [airport](https://www.pexels.com/video/856984/), [office](https://www.pexels.com/video/6914684/). Each camera is asked for its own things, and the first three seconds of each are used.

In [ ]:
import urllib.request
import cv2
import numpy as np

PROMPTS = {"856984": ["airplane", "person"], "6914684": ["person", "glasses"]}  # Pexels id -> what to find
BASE = "https://github.com/aryamandarda/tamserve/releases/download/v0.2.0/"
N = 90  # frames per camera: three seconds


def frames_of(pexels_id):
    """The clip's first N frames as 1280 x 720 RGB arrays, as a camera would send them."""
    path = f"/tmp/{pexels_id}.mp4"
    urllib.request.urlretrieve(f"{BASE}{pexels_id}.mp4", path)
    capture, frames = cv2.VideoCapture(path), []
    while len(frames) < N and (frame := capture.read())[0]:
        frames.append(cv2.cvtColor(frame[1], cv2.COLOR_BGR2RGB))
    return frames


videos = {pexels_id: frames_of(pexels_id) for pexels_id in PROMPTS}
print({pexels_id: len(frames) for pexels_id, frames in videos.items()})

One tracker, one call per round. SAM 3 looks at each camera now and then and outlines what was asked for; EdgeTAM, inside the engine, follows those outlines in between. Each thing gets a number, and `tracker.names()` says which prompt it was found for.

A look is the slow part, and the free T4 is an old GPU: a look takes about 3 seconds there, against 0.07 on an RTX 4090. So this notebook looks every sixth frame. The default is every third.

In [ ]:
import time
from tamserve import Engine, Tracker, build_video_predictor

tracker = Tracker(Engine(build_video_predictor("cuda")), PROMPTS, look_every=6)  # loads EdgeTAM, then SAM 3
found = {pexels_id: [] for pexels_id in videos}  # per frame: {number: (prompt, mask)}
start = time.perf_counter()
for k in range(N):
    for pexels_id, objects in tracker.step({pexels_id: v[k] for pexels_id, v in videos.items()}).items():
        names = tracker.names(pexels_id)
        found[pexels_id].append({t: (names[t], m[::2, ::2].numpy()) for t, m in objects.items()})  # half size, for drawing
took = time.perf_counter() - start
for pexels_id in videos:
    print(pexels_id, {t: name for t, (name, _) in sorted(found[pexels_id][-1].items())})
print(f"{len(videos) * N} frames from {len(videos)} cameras in {took:.0f} s on {torch.cuda.get_device_name()}")

In [ ]:
import subprocess
from IPython.display import Video

COLORS = {"airplane": (65, 160, 255), "person": (255, 99, 71), "glasses": (255, 200, 40)}
writer = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", "1280x360",
                           "-r", "30", "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "/tmp/named.mp4"],
                          stdin=subprocess.PIPE)
for k in range(N):
    wall = np.zeros((360, 1280, 3), np.uint8)
    for x, pexels_id in enumerate(videos):
        tile = cv2.resize(videos[pexels_id][k], (640, 360), interpolation=cv2.INTER_AREA).astype(np.float32)
        for t, (name, m) in found[pexels_id][k].items():
            m = cv2.resize(m.astype(np.uint8), (640, 360), interpolation=cv2.INTER_NEAREST).astype(bool)
            tile[m] = tile[m] * 0.5 + np.array(COLORS[name]) * 0.5
        tile = np.ascontiguousarray(tile.astype(np.uint8))
        for t, (name, m) in found[pexels_id][k].items():
            ys, xs = np.nonzero(m)
            if len(xs):
                cv2.putText(tile, f"{name} {t}", (int(xs.mean()) - 40, int(ys.min()) + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        wall[:, x * 640:(x + 1) * 640] = tile
    writer.stdin.write(wall.tobytes())
writer.stdin.close()
writer.wait()
Video("/tmp/named.mp4", embed=True, width=960)

Next: cameras that do not move (`still=True`), a larger model for more accuracy, and `PeopleTracker`, which follows people for a fraction of the cost, in the [README](https://github.com/aryamandarda/tamserve#following-what-you-name).